In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_customer_survey")

In [0]:
display(df.limit(5))

In [0]:
row_count = df.count()
row_count

In [0]:
print("\nDUPLICATE ANALYSIS")
duplicate_survey_ids = df.groupBy("survey_id").count().filter(col("count") > 1)
print(f"Duplicate survey_ids: {duplicate_survey_ids.count()}")

In [0]:
print("\nNULL VALUE ANALYSIS")

null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ])
print("Null counts by column:")
display(null_counts)

In [0]:
df = df.withColumn(
    "responded_flag",
    when(col("survey_response_date").isNotNull(), f.lit(True)).otherwise(f.lit(False))
)
display(df)

In [0]:
df_cleaned = df.withColumn('survey_sent_date', f.date_format(col('survey_sent_date'), 'yyyy-MM-dd'))
df_cleaned = df_cleaned.withColumn('survey_response_date', f.date_format(col('survey_response_date'), 'yyyy-MM-dd'))
display(df_cleaned.limit(5))

In [0]:
df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customer_survey")